# Text generation

Let's recode a [nice blog post](https://huggingface.co/blog/how-to-generate) from Hugging Face on text generation to understand the key concepts.

## Installation

The [`transformers`](https://github.com/huggingface/transformers) library developed by the French-American startup [Hugging Face](https://huggingface.co/) shares a lot of high quality state of the art pretrained NLP models.

It is easily installable with `pip`.

In [ ]:
!pip install 'transformers < 4.3'

## Language

We are going to see how to generate English text, but if you change the value here the generated text will be in French: the approach is not language dependent. The only important thing is the content of the pretraining data.

In [ ]:
lang = "en"
# lang = "fr"

## Getting a pretrained model

In [ ]:
import functools
import typing

import numpy
import tensorflow
import transformers
import tqdm.notebook


pretraining_name = "gpt2" if lang == "en" else "antoiloui/belgpt2"

model = transformers.TFGPT2LMHeadModel.from_pretrained(pretraining_name)
tokenizer = transformers.GPT2Tokenizer.from_pretrained(pretraining_name)

## The generation mechanism

We are maximizing $P(\text{Text}| \text{Prompt})$ with the following decomposition:

$$P(\text{Text}| \text{Prompt}) = \Pi_{i=1}^{\text{Text length}} P(\text{Word}_i| \text{Word}_{j<i}, \text{Prompt})$$

The key is to find the maximal value of this product. It's intractable to compute exactly. Our goals is to find the best approximation possible.

## Tokenization

When we downloaded a pretrained model, we also got a tokenizer to use with it. Let's produce functions to encode/decode text to/from the token space.

In [ ]:
def encode(sentence: str) -> tensorflow.Tensor:
  tokens = tokenizer.encode(sentence,
                            add_special_tokens=False,
                            return_tensors="tf")
  bos = tensorflow.constant([[model.config.bos_token_id]], dtype="int64")
  return tensorflow.concat([bos, tensorflow.cast(tokens, "int64")], axis=1)


def decode(tokens: tensorflow.Tensor) -> str:
  if tokens.ndim == 2:
    tokens = tensorflow.squeeze(tokens)
  return tokenizer.decode(tokens[1:], clean_up_tokenization_spaces=True)

## Shape of the model output

In [ ]:
def describe_shapes(input_string: str,
                    output_tensor: tensorflow.Tensor,
                    output_past: typing.List[tensorflow.Tensor]
                   ) -> str:
  print("-" * 80)
  print(f"Output shape for input {input_string!r}:", output_tensor.shape)
  print(f"Size of the past:", len(output_past))
  print(f"Type of the past:", type(output_past[0]))


empty_input = ""
descartes_input = ("I think, therefore I am"
                   if lang == "en"
                   else "Je pense, donc je suis")

empty_output, empty_past = model(encode(empty_input),
                                 return_dict=False)
descartes_output, descartes_past = model(encode(descartes_input),
                                         return_dict=False)

describe_shapes(empty_input, empty_output, empty_past)
describe_shapes(descartes_input, descartes_output, descartes_past)

## Nicer forward function

To ease future decoding, we will use a function that always returns only the last item of its second dimension: it's the prediction of the model. The rest is actually not important to us. It will also return the `past` element of the model output to save on future computations.

In [ ]:
def forward(tokens: tensorflow.Tensor,
            past: typing.Optional[typing.List[tensorflow.Tensor]] = None
           ) -> typing.Tuple[tensorflow.Tensor, typing.List[tensorflow.Tensor]]:
  logits, new_past = model(tokens, past=past, return_dict=False)
  return logits[:, -1, :], new_past

Note that when the `past` argument is used, we must not give the past words to the model. We have to pick either `forward(all_tokens)` or `forward(last_token, past)`.

where `all_tokens` would for example be a sequence of 7 tokens of shape `(1, 7, 50257)` and where `last_token` would be of shape `(1, 1, 50257)`.

## Decoding loop

A decoding loop has the following form:

1. Encode everything that was produced up to now
2. Produce the most likely next token
3. Repeat step 1. and 2. until a stopping criterion is met. In this demonstration we will just stop after a fixed number of words.

The key step is the step 2. We will code the other steps in a function to focus on step 2 for the rest of the demonstration.

In [ ]:
def decoding_loop(step_function) -> str:

  @functools.wraps(step_function)
  def wrapper(prompt, length, *step_args, **step_kwargs):
    token_ids = encode(prompt)
    past = None
    decoded = [token_ids]
    for i in tqdm.notebook.trange(length, desc="Words", leave=False):
      logits, past = forward(decoded[-1], past)
      index = step_function(logits, *step_args, **step_kwargs)
      decoded += [index[None, None]]
    decoded_tensor = tensorflow.concat(decoded, axis=1)
    return decode(decoded_tensor)

  return wrapper

## Greedy decoding

For a first approach, we will just pick the highest value each time. This is called greedy decoding and is very bad in general. We'll improve afterwards but this is a good way to finish setting up the pipeline.

Here is the example from the blog post mentioned at the start of the demonstration:

![Greedy decoding](https://huggingface.co/blog/assets/02_how-to-generate/greedy_search.png)

In [ ]:
@decoding_loop
def greedy(logits: tensorflow.Tensor) -> tensorflow.Tensor:
  return tensorflow.math.argmax(logits, axis=1)[0]


print(greedy("I went to the", 10))

## Test on some inputs

In [ ]:
if lang == "en":
    inputs = ["I went to the",
              "I am feeling",
              "How do you",
              "How comes that",
              "I'm fine thank you, you should"]
else:
    inputs = ["Je suis allé au",
              "Je me sens",
              "Comment vas-tu",
              "Comment se fait-il que",
              "Ça va merci, tu devrais"]

In [ ]:
def test_decoding(function, *args, **kwargs) -> None:
  generations = []
  for i in tqdm.notebook.tqdm(inputs, desc="Prompts", leave=False):
    generations.append(function(i, *args, **kwargs))
  for prompt, generation in zip(inputs, generations):
    print("—" * 80)
    print("Prompt:", prompt)
    print("Generation:", generation)
  print("—" * 80)

test_decoding(greedy, 50)

Answers usually “loop” quickly. It's a usual problem with greedy decoding.

## Sampling

Let's sample from the softmax distribution instead of decoding greedily to avoid looping.

In [ ]:
@decoding_loop
def sampling(logits: tensorflow.Tensor) -> tensorflow.Tensor:
  softmaxed = tensorflow.nn.log_softmax(logits)
  return tensorflow.random.categorical(softmaxed, 1)[0][0]

test_decoding(sampling, 50)

## Top-k sampling

To avoid very uncommon words, we can restrict our sampling to the top $k$ words.

In [ ]:
@decoding_loop
def k_sampling(logits: tensorflow.Tensor, k: int) -> tensorflow.Tensor:
  topk = tensorflow.math.top_k(logits, k=k)
  softmaxed = tensorflow.nn.log_softmax(topk.values)
  sampled = tensorflow.random.categorical(softmaxed, 1)[0][0]
  return tensorflow.cast(topk.indices[0, sampled], "int64")

test_decoding(k_sampling, 50, k=20)

## Top-p sampling

To restrict the sampling in an even more robust way, we can also use less than $k$ words if we already consumed more than $p$ probability mass.

In [ ]:
@decoding_loop
def k_p_sampling(logits: tensorflow.Tensor, k: int, p: float
                ) -> tensorflow.Tensor:
    topk = tensorflow.math.top_k(logits, k=k)
    softmaxed = tensorflow.nn.softmax(topk.values)
    current_proba_sum = 0
    j = 0
    while current_proba_sum < p:
      current_proba_sum += softmaxed[0, j]
      j += 1
    sampled = tensorflow.random.categorical(
        tensorflow.math.log(softmaxed[:, :j]), 1)[0][0]
    return tensorflow.cast(topk.indices[0, sampled], "int64")

test_decoding(k_p_sampling, length=100, k=20, p=0.85)

## Direct usage of the `transformers` library

If you need text generation in the future, you can use the library directly. It's also able to tackle a few more [tasks](https://huggingface.co/transformers/task_summary.html).

In [ ]:
inputs = ["The NASA just tweeted:"]

In [ ]:
def transformers_generate(prompt: str,
                          length: int,
                          temperature: float,
                          top_k: int,
                          top_p: float) -> str:
  token_ids = encode(prompt)
  generated = model.generate(input_ids=tensorflow.cast(token_ids, "int32"),
                             max_length=len(token_ids) + length,
                             temperature=temperature,
                             top_k=top_k,
                             top_p=top_p,
                             do_sample=True,
                             num_return_sequences=1)
  return decode(tensorflow.cast(generated, "int64"))

test_decoding(transformers_generate, length=1000, temperature=1, top_k=30, top_p=0.95)